In [2]:
pip install reportlab

STEP 1 — IMPORT LIBRARIES


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Image,
    PageBreak
)

from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter

STEP 2 — LOAD DATASET

In [4]:
df = pd.read_csv("SuperStoreOrders.csv")
df.head()

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
0,AG-2011-2040,1/1/2011,6/1/2011,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,Office Supplies,Storage,"Tenex Lockers, Blue",408,2,0.0,106.140,35.46,Medium,2011
1,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Office Supplies,Supplies,"Acme Trimmer, High Speed",120,3,0.1,36.036,9.72,Medium,2011
2,HU-2011-1220,1/1/2011,5/1/2011,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,...,Office Supplies,Storage,"Tenex Box, Single Width",66,4,0.0,29.640,8.17,High,2011
3,IT-2011-3647632,1/1/2011,5/1/2011,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,...,Office Supplies,Paper,"Enermax Note Cards, Premium",45,3,0.5,-26.055,4.82,High,2011
4,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Furniture,Furnishings,"Eldon Light Bulb, Duo Pack",114,5,0.1,37.770,4.70,Medium,2011


 STEP 3 — DATA UNDERSTANDING


In [5]:
print(df.head())

print(df.info())

print(df.isnull().sum())

          order_id order_date ship_date       ship_mode    customer_name  \
0     AG-2011-2040   1/1/2011  6/1/2011  Standard Class  Toby Braunhardt   
1    IN-2011-47883   1/1/2011  8/1/2011  Standard Class      Joseph Holt   
2     HU-2011-1220   1/1/2011  5/1/2011    Second Class    Annie Thurman   
3  IT-2011-3647632   1/1/2011  5/1/2011    Second Class     Eugene Moren   
4    IN-2011-47883   1/1/2011  8/1/2011  Standard Class      Joseph Holt   

       segment            state    country  market   region  ...  \
0     Consumer      Constantine    Algeria  Africa   Africa  ...   
1     Consumer  New South Wales  Australia    APAC  Oceania  ...   
2     Consumer         Budapest    Hungary    EMEA     EMEA  ...   
3  Home Office        Stockholm     Sweden      EU    North  ...   
4     Consumer  New South Wales  Australia    APAC  Oceania  ...   

          category sub_category                 product_name sales quantity  \
0  Office Supplies      Storage          Tenex Lockers,

# STEP 4 — DATA CLEANING

In [6]:
# Remove duplicate rows
df = df.drop_duplicates()

# Convert sales to numeric (before filling NaNs)
df['sales'] = pd.to_numeric(df['sales'], errors='coerce')

# Fill missing NaNs specifically in the 'sales' column with 0
df['sales'] = df['sales'].fillna(0)

# Ensure 'sales' column is float type
df['sales'] = df['sales'].astype(float)

# Convert dates
df["order_date"] = pd.to_datetime(
    df["order_date"],
    dayfirst=True,
    format='mixed'
)

# Create month column
df["month"] = df["order_date"].dt.month_name()

# STEP 5 — BUSINESS ANALYSIS

In [7]:
# Ensure sales and profit are numeric before calculations
df['sales'] = pd.to_numeric(df['sales'], errors='coerce').fillna(0)
df['profit'] = pd.to_numeric(df['profit'], errors='coerce').fillna(0)

# Total Sales
total_sales = round(df["sales"].sum(), 2)

# Total Profit
total_profit = round(df["profit"].sum(), 2)

# Total Orders
total_orders = df["order_id"].nunique()

# Category Analysis
category_sales = (
    df.groupby("category")["sales"]
    .sum()
    .sort_values(ascending=False)
)

# Region Profit
region_profit = (
    df.groupby("region")["profit"]
    .sum()
    .sort_values(ascending=False)
)

# Market Analysis
market_sales = (
    df.groupby("market")["sales"]
    .sum()
    .sort_values(ascending=False)
)

# Monthly Sales
monthly_sales = (
    df.groupby("month")["sales"]
    .sum()
)

# Top Countries
top_countries = (
    df.groupby("country")["sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

# STEP 6 — CREATE CHARTS

In [8]:
# CATEGORY SALES


plt.figure(figsize=(8,5))

category_sales.plot(kind="bar")

plt.title("Sales by Category")
plt.xlabel("Category")
plt.ylabel("Sales")

plt.tight_layout()

plt.savefig("category_sales.png")

plt.close()

In [9]:
# REGION PROFIT

plt.figure(figsize=(8,5))

sns.barplot(
    x=region_profit.index,
    y=region_profit.values
)

plt.title("Profit by Region")
plt.xlabel("Region")
plt.ylabel("Profit")

plt.tight_layout()

plt.savefig("region_profit.png")

plt.close()


In [10]:
# MARKET SALES

plt.figure(figsize=(8,5))

market_sales.plot(kind="bar")

plt.title("Sales by Market")
plt.xlabel("Market")
plt.ylabel("Sales")

plt.tight_layout()

plt.savefig("market_sales.png")

plt.close()

In [11]:
# MONTHLY SALES TREND

plt.figure(figsize=(12,5))

monthly_sales.plot(kind="line", marker="o")

plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")

plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig("monthly_sales.png")

plt.close()

In [12]:
# TOP COUNTRIES

plt.figure(figsize=(10,5))

sns.barplot(
    x=top_countries.index,
    y=top_countries.values
)

plt.title("Top 10 Countries by Sales")
plt.xlabel("Country")
plt.ylabel("Sales")

plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig("top_countries.png")

plt.close()


# STEP 7 — CREATE PDF REPORT

In [13]:
doc = SimpleDocTemplate(
    "Global_Superstore_Report.pdf",
    pagesize=letter
)

styles = getSampleStyleSheet()

content = []

In [14]:
# COVER PAGE


title = Paragraph(
    "Global Superstore Sales and Profitability Analysis",
    styles['Title']
)

content.append(title)

content.append(Spacer(1, 20))

intro = Paragraph(
    """
    This report presents a comprehensive business analysis
    of global Superstore order data using Python-based
    data analysis and visualization techniques.

    The analysis focuses on sales performance,
    profitability trends, regional performance,
    and category-level business insights.
    """,
    styles['BodyText']
)

content.append(intro)

content.append(PageBreak())

In [15]:
# EXECUTIVE SUMMARY

summary_title = Paragraph(
    "Executive Summary",
    styles['Heading1']
)

content.append(summary_title)

summary_text = Paragraph(
    f"""
    The analysis identified strong global sales performance
    across multiple markets and product categories.

    Technology products generated the highest overall
    revenue contribution, while regional profitability
    analysis highlighted stronger operational performance
    within selected business regions.

    The dataset was cleaned and standardized to improve
    reporting accuracy and analytical consistency.

    Total Sales: ${total_sales}

    Total Profit: ${total_profit}

    Total Orders Processed: {total_orders}
    """,
    styles['BodyText']
)

content.append(summary_text)

content.append(Spacer(1, 20))

In [16]:
# DATA CLEANING


cleaning_title = Paragraph(
    "Data Cleaning and Preparation",
    styles['Heading1']
)

content.append(cleaning_title)

cleaning_text = Paragraph(
    """
    Several preprocessing steps were performed
    before conducting business analysis.

    Data preparation activities included:
    - duplicate record removal
    - handling missing values
    - date standardization
    - validation of numerical fields
    - formatting consistency improvements

    These processes improved data quality
    and ensured more reliable business reporting.
    """,
    styles['BodyText']
)

content.append(cleaning_text)

content.append(Spacer(1, 20))


In [17]:
# CATEGORY ANALYSIS

category_title = Paragraph(
    "Category Performance Analysis",
    styles['Heading1']
)

content.append(category_title)

category_text = Paragraph(
    """
    Technology products generated the highest
    revenue contribution during the analysis period,
    indicating strong market demand and customer preference.

    Furniture products maintained stable sales volume;
    however, profit margins were comparatively lower,
    suggesting opportunities for pricing optimization
    and cost management improvements.
    """,
    styles['BodyText']
)

content.append(category_text)

content.append(Spacer(1, 10))

content.append(Image("category_sales.png", width=400, height=250))

content.append(Spacer(1, 20))

In [18]:
# REGIONAL ANALYSIS


region_title = Paragraph(
    "Regional Profitability Analysis",
    styles['Heading1']
)

content.append(region_title)

region_text = Paragraph(
    """
    Regional analysis demonstrated stronger profitability
    performance within high-demand business regions.

    Some regions achieved healthy sales volume but lower
    profit margins, indicating potential operational
    inefficiencies or pricing challenges.

    The findings highlight opportunities for improved
    regional sales strategy and cost optimization.
    """,
    styles['BodyText']
)

content.append(region_text)

content.append(Spacer(1, 10))

content.append(Image("region_profit.png", width=400, height=250))

content.append(Spacer(1, 20))


In [19]:
# MARKET ANALYSIS


market_title = Paragraph(
    "Market Performance Analysis",
    styles['Heading1']
)

content.append(market_title)

market_text = Paragraph(
    """
    Market-level analysis revealed strong sales contribution
    from key international markets.

    Sales distribution across markets indicates
    diversified revenue generation and broad
    customer demand across multiple geographic regions.
    """,
    styles['BodyText']
)

content.append(market_text)

content.append(Spacer(1, 10))

content.append(Image("market_sales.png", width=400, height=250))

content.append(Spacer(1, 20))

In [20]:
# MONTHLY SALES ANALYSIS


monthly_title = Paragraph(
    "Monthly Sales Trend Analysis",
    styles['Heading1']
)

content.append(monthly_title)

monthly_text = Paragraph(
    """
    Monthly sales analysis identified seasonal
    fluctuations in customer purchasing behavior.

    Higher sales activity during selected months
    suggests increased demand periods and potential
    seasonal sales opportunities.

    Continuous monitoring of monthly trends can support
    more effective inventory planning and demand forecasting.
    """,
    styles['BodyText']
)

content.append(monthly_text)

content.append(Spacer(1, 10))

content.append(Image("monthly_sales.png", width=450, height=250))

content.append(Spacer(1, 20))

In [21]:
# COUNTRY ANALYSIS


country_title = Paragraph(
    "Top Country Sales Analysis",
    styles['Heading1']
)

content.append(country_title)

country_text = Paragraph(
    """
    Country-level analysis demonstrated concentrated
    revenue generation within high-performing international markets.

    The analysis highlights strong customer demand
    across selected countries and provides insight into
    global sales distribution patterns.
    """,
    styles['BodyText']
)

content.append(country_text)

content.append(Spacer(1, 10))

content.append(Image("top_countries.png", width=450, height=250))

content.append(Spacer(1, 20))

In [22]:
# RECOMMENDATIONS


recommendation_title = Paragraph(
    "Business Recommendations",
    styles['Heading1']
)

content.append(recommendation_title)

recommendation_text = Paragraph(
    """
    Based on the analysis findings,
    the following business recommendations are suggested:

    1. Increase strategic focus on high-performing
    product categories to maximize revenue growth.

    2. Review pricing strategies within lower-margin
    categories to improve profitability performance.

    3. Expand market presence within high-performing regions
    to strengthen global revenue contribution.

    4. Utilize seasonal sales trends to improve
    inventory planning and operational forecasting.

    5. Monitor regional profitability metrics regularly
    to support data-driven business decision-making.
    """,
    styles['BodyText']
)

content.append(recommendation_text)

content.append(Spacer(1, 20))

In [23]:
# CONCLUSION

conclusion_title = Paragraph(
    "Conclusion",
    styles['Heading1']
)

content.append(conclusion_title)

conclusion_text = Paragraph(
    """
    This analysis successfully identified key sales trends,
    profitability drivers, and business growth opportunities
    across multiple markets and product categories.

    The findings from this report can support
    strategic business planning, operational optimization,
    and long-term performance improvement initiatives.

    Continuous monitoring of sales and profitability metrics
    is recommended to maintain sustainable business growth.
    """,
    styles['BodyText']
)

content.append(conclusion_text)

In [24]:
# BUILD PDF


doc.build(content)

print("Professional PDF Report Created Successfully")

Professional PDF Report Created Successfully
